In [1]:
import sys, json
import pandas as pd  # para las tablas
sys.path.append("../src")

from weather_api import get_weather

# cargo las actividades que ya baje (asi no llamo a strava cada vez)
with open("../data/raw/activities.json", encoding="utf-8") as f:
    acts = json.load(f)

In [2]:
# paso las actividades a tabla para verlas
df = pd.json_normalize(acts)
print(df.shape)  # filas y columnas
df.head()        # las 5 primeras

(537, 64)


,resource_state,name,distance,moving_time,elapsed_time,total_elevation_gain,type,sport_type,workout_type,device_name,...,athlete.resource_state,map.id,map.summary_polyline,map.resource_state,average_cadence,average_watts,max_watts,weighted_average_watts,device_watts,kilojoules
0,2,Hybrid,0.0,3025,3025,0.0,Workout,Workout,NaN,Suunto Run,...,1,a20780762632,,2,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Carrera de tarde,8008.0,2706,2728,32.0,Run,Run,NaN,Suunto Run,...,1,a20764750642,uxpuFrfsUW|AGx@Gb@e@|AUlAWt@c@~@g@bB[f@[|@i@rA...,2,88.4,216.7,283.0,212.0,True,586.3
2,2,Entrenamiento con pesas matutino,0.0,3610,3610,0.0,WeightTraining,WeightTraining,30.0,Suunto Run,...,1,a20747167484,,2,NaN,NaN,NaN,NaN,NaN,NaN
3,2,Entrenamiento con pesas matutino,0.0,3601,3601,0.0,WeightTraining,WeightTraining,30.0,Suunto Run,...,1,a20727021925,,2,NaN,NaN,NaN,NaN,NaN,NaN
4,2,Pata,0.0,3661,3661,0.0,WeightTraining,WeightTraining,30.0,Suunto Run,...,1,a20671053552,,2,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df[["name", "sport_type", "start_date_local", "distance", "moving_time", "gear_id"]].head(10)

,name,sport_type,start_date_local,distance,moving_time,gear_id
0,Hybrid,Workout,2026-10-29T17:04:03Z,0.0,3025,NaN
1,Carrera de tarde,Run,2026-10-28T13:12:16Z,8008.0,2706,g28164179
2,Entrenamiento con pesas matutino,WeightTraining,2026-10-27T07:51:31Z,0.0,3610,NaN
3,Entrenamiento con pesas matutino,WeightTraining,2026-10-23T07:02:13Z,0.0,3601,NaN
4,Pata,WeightTraining,2026-10-21T07:12:10Z,0.0,3661,NaN
5,Hybrid,Workout,2026-10-19T16:46:43Z,0.0,3014,NaN
6,Entrenamiento matutino,Workout,2026-10-16T10:02:50Z,0.0,2857,NaN
7,Morning Weight Training,WeightTraining,2026-10-15T05:45:52Z,0.0,2426,NaN
8,Carrera de tarde,Run,2026-10-08T17:23:52Z,7009.0,2217,g28164179
9,Entrenamiento con pesas matutino,WeightTraining,2026-10-07T07:08:49Z,0.0,3168,NaN


In [4]:
# me quedo con las carreras que tienen gps
runs = df[(df["sport_type"] == "Run") & (df["start_latlng"].str.len() == 2)].copy()

# lat y lon redondeadas a 1 decimal (unos 10 km) para agrupar por zona
runs["lat"] = runs["start_latlng"].str[0].round(1)
runs["lon"] = runs["start_latlng"].str[1].round(1)

# hora de inicio en punto, para cruzarla luego con el clima
runs["hora"] = pd.to_datetime(runs["start_date_local"].str[:19]).dt.floor("h")

zonas = runs.groupby(["lat", "lon"])["hora"].agg(["min", "max", "count"]).reset_index()
print(len(runs), "carreras con gps")
print(len(zonas), "zonas distintas")
zonas.sort_values("count", ascending=False).head(10)

309 carreras con gps
18 zonas distintas


,lat,lon,min,max,count
13,40.4,-3.7,2023-12-17 09:00:00,2026-10-28 13:00:00,136
9,40.3,-3.9,2020-09-02 19:00:00,2026-04-29 18:00:00,110
11,40.4,-3.9,2021-07-04 19:00:00,2025-05-26 18:00:00,17
12,40.4,-3.8,2024-03-17 09:00:00,2026-06-16 19:00:00,16
17,43.7,-8.1,2024-08-13 07:00:00,2024-08-16 18:00:00,6
14,40.5,-3.7,2023-11-12 09:00:00,2025-11-23 09:00:00,5
10,40.3,-3.7,2024-01-21 14:00:00,2025-05-09 18:00:00,4
4,37.2,-7.2,2025-08-06 19:00:00,2025-08-09 09:00:00,3
2,36.6,-4.5,2025-08-12 07:00:00,2025-08-14 07:00:00,2
7,40.3,-4.6,2022-05-01 10:00:00,2024-04-28 08:00:00,2


In [7]:
from datetime import date, timedelta

# open-meteo tarda unos dias en tener el historico, asi que pido hasta hace 5 dias
limite = date.today() - timedelta(days=5)

climas = []
for _, z in zonas.iterrows():
    inicio = z["min"].date()
    fin = min(z["max"].date(), limite)
    # si la zona solo tiene carreras muy recientes no hay historico todavia, la salto
    if inicio > fin:
        print(z["lat"], z["lon"], "sin historico todavia, la salto")
        continue
    c = get_weather(z["lat"], z["lon"], inicio.strftime("%Y-%m-%d"), fin.strftime("%Y-%m-%d"))
    climas.append(c)
    print(z["lat"], z["lon"], len(c), "horas")

10.1 99.8 24 horas
13.7 100.5 24 horas
36.6 -4.5 72 horas
36.8 -3.9 24 horas
37.2 -7.2 96 horas
37.4 -6.0 24 horas
39.2 -0.2 24 horas
40.3 -4.6 17496 horas
40.3 -4.5 24 horas
40.3 -3.9 49584 horas
40.3 -3.7 11400 horas
40.4 -3.9 34152 horas
40.4 -3.8 19728 horas
40.4 -3.7 25032 horas
40.5 -3.7 17832 horas
43.3 -2.0 24 horas
43.4 -3.5 24 horas
43.7 -8.1 96 horas


In [6]:
# renombro time a hora para que coincida con runs
clima = clima.rename(columns={"time": "hora"})

# redondeo lat y lon en los dos lados para que el merge no falle por decimales
clima["lat"] = clima["lat"].round(1)
clima["lon"] = clima["lon"].round(1)

runs_clima = runs.merge(clima, on=["lat", "lon", "hora"], how="left")
print(runs_clima["temperature_2m"].isna().sum(), "carreras sin clima")
runs_clima[["name", "hora", "distance", "temperature_2m", "relative_humidity_2m", "wind_speed_10m"]].head(10)

1 carreras sin clima


,name,hora,distance,temperature_2m,relative_humidity_2m,wind_speed_10m
0,Carrera de tarde,2026-10-28 13:00:00,8008.0,NaN,NaN,NaN
1,Carrera de tarde,2026-10-08 17:00:00,7009.0,18.7,36.0,18.6
2,FULL MOON RUN CLUB,2026-10-03 11:00:00,7135.0,27.9,53.0,3.8
3,Carrera de noche,2026-09-23 19:00:00,6008.0,31.0,15.0,6.0
4,Evening Run,2026-09-19 19:00:00,7004.0,27.0,28.0,5.7
5,Evening Run,2026-09-10 19:00:00,4458.0,25.9,14.0,8.4
6,Carrera de mañana,2026-09-06 08:00:00,6069.0,25.7,33.0,5.4
7,Carrera de mañana,2026-08-30 09:00:00,6003.0,20.6,39.0,1.8
8,Carrera de noche,2026-08-24 19:00:00,6012.0,25.0,39.0,19.5
9,Carrera de mañana,2026-08-15 08:00:00,5011.0,22.5,57.0,6.8
